# #272 validation: BEIR non-regression gate on hybrid default

Issue #272 swapped sqlite-vec's default L2 metric for `distance_metric=cosine`
and rebuilt the `vec_chunks` table in-place on v1 DBs.  Locally the full
5-dataset BEIR regression takes ~1h45m on CPU; on Colab T4 it drops to
~30-45 min.  This notebook runs the canonical bench on the feature
branch and compares against the frozen v1 baseline at
`experiments/results/beir_adaptive_baseline.json`.

Pass criterion: **every dataset within `REGRESSION_TOLERANCE = 0.005`
absolute NDCG@10 of baseline**, matching the pytest gate in
`tests/test_beir_regression.py`.

Model is **plain BGE-small-en-v1.5** (not the tuned v2/v3) because the
baseline was established on the unmodified pipeline.  Tuned-model
validation lives in `beir_benchmark_colab.ipynb`.

In [ ]:
# Cell 1: Setup.  Clone the #272 feature branch.
#
# Colab kernels sometimes inherit a stale cwd after a previous rm -rf,
# which breaks every subsequent shell magic.  Force the kernel back to
# a known-good directory before any `!` command runs.
import os

os.chdir("/")
os.chdir("/content")

!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch feature/272-vec0-cosine-metric https://github.com/stffns/vstash.git /content/vstash

os.chdir("/content/vstash")
!pip install -q -e .
print("cwd:", os.getcwd())

# Sanity check: the vec_chunks DDL in store.py must declare cosine.
# If this assert fires, we cloned the wrong branch or a stale cache.
with open("vstash/store.py") as f:
    src = f.read()
assert "distance_metric=cosine" in src, "branch does not contain the #272 fix"
assert 'SCHEMA_VERSION = "2"' in src, "branch is not at schema v2"
print("#272 markers present in store.py: OK")

In [ ]:
# Cell 2: Run the 5-dataset BEIR bench on BGE-small (paper baseline model).
# Runtime on T4: ~30-45 min, dominated by fiqa (57k docs) embedding.
import os
from pathlib import Path

os.chdir("/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

# Clear any stale benchmark JSON so Cell 3 cannot accidentally read a
# pre-run artefact shipped with the repo.
stale = Path("experiments/results/beir_benchmark.json")
if stale.exists():
    stale.unlink()
    print(f"  cleared stale results file: {stale}")

!python -m experiments.beir_benchmark --no-chroma --model BAAI/bge-small-en-v1.5

In [ ]:
# Cell 3: Compare against the frozen v1 baseline and report pass/fail.
# Same criterion as tests/test_beir_regression.py (absolute NDCG@10
# within REGRESSION_TOLERANCE of baseline).
import json
from pathlib import Path

REGRESSION_TOLERANCE = 0.005

results_path = Path("experiments/results/beir_benchmark.json")
baseline_path = Path("experiments/results/beir_adaptive_baseline.json")
assert results_path.exists(), "beir_benchmark.json not found. Cell 2 must complete first."
assert baseline_path.exists(), "baseline JSON missing -- repo is incomplete."

results = json.loads(results_path.read_text())
baseline = json.loads(baseline_path.read_text())

assert results["model"] == "BAAI/bge-small-en-v1.5", (
    f"Cell 2 must be run on BGE-small to compare against the paper baseline, got {results['model']!r}"
)

print(f"Model:              {results['model']}")
print(f"Baseline timestamp: {baseline.get('timestamp', 'unknown')}")
print(f"Tolerance:          +/- {REGRESSION_TOLERANCE:.3f} absolute NDCG@10")
print()
print(f"{'Dataset':<10} {'Baseline':>10} {'Actual':>10} {'Delta':>10}  {'Verdict':<12}")
print("-" * 60)

failures: list[tuple[str, float, float]] = []
actual_by_dataset = {r["dataset"]: r["vstash"]["ndcg_10"] for r in results["results"]}

for dataset, bvals in baseline["results"].items():
    expected = bvals["ndcg_10"]
    actual = actual_by_dataset.get(dataset)
    if actual is None:
        print(f"{dataset:<10} {expected:>10.4f}     (skipped by Cell 2)")
        continue
    delta = actual - expected
    verdict = "PASS" if delta >= -REGRESSION_TOLERANCE else "FAIL"
    if verdict == "FAIL":
        failures.append((dataset, expected, actual))
    print(f"{dataset:<10} {expected:>10.4f} {actual:>10.4f} {delta:>+10.4f}  {verdict:<12}")

print()
if failures:
    print(f"REGRESSION: {len(failures)} dataset(s) below tolerance.")
    for ds, exp, act in failures:
        print(f"  {ds}: {act:.4f} < {exp:.4f} - {REGRESSION_TOLERANCE}")
    raise SystemExit(1)
else:
    print("GATE PASS: all 5 BEIR datasets within tolerance.  #272 cosine migration is safe to merge.")

In [ ]:
# Cell 4 (optional): run the pytest benchmark suite directly.
# Exercises the same numbers via the pytest gate in tests/test_beir_regression.py,
# including the "adaptive beats fixed on all datasets" check.  Reuses the
# cached BEIR data from Cell 2, so no additional download.
import os

os.chdir("/content/vstash")
!python -m pytest tests/test_beir_regression.py -v -m benchmark